In [1]:
#기본 라이브러리
import torch
import json
import re
import json
import os

from tqdm import tqdm 

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
#공통 변수 먼저 설정
BASE_PATH = "/gpfs/data/oermannlab/gaifl/users/yl14814/medqa_hybrid_project"

MODEL_NAME = "llama31"
DATASET_NAME = "MedQA"

BASELINE_RESULT_DIR = f"{BASE_PATH}/03_results/baselines/{MODEL_NAME}/{DATASET_NAME}"
DEBUG_RESULT_DIR = f"{BASE_PATH}/03_results/debug/{MODEL_NAME}/{DATASET_NAME}"

os.makedirs(BASELINE_RESULT_DIR, exist_ok=True)
os.makedirs(DEBUG_RESULT_DIR, exist_ok=True)

print("Baseline path:", BASELINE_RESULT_DIR)
print("Debug path:", DEBUG_RESULT_DIR)

Baseline path: /gpfs/data/oermannlab/gaifl/users/yl14814/medqa_hybrid_project/03_results/baselines/llama31/MedQA
Debug path: /gpfs/data/oermannlab/gaifl/users/yl14814/medqa_hybrid_project/03_results/debug/llama31/MedQA


In [3]:
#BigBio MedQA Load
dataset = load_dataset(
    "bigbio/med_qa",
    "med_qa_en_4options_bigbio_qa",
    trust_remote_code=True
)

test_data = dataset["test"]

print("Test size:", len(test_data))
print(test_data[0])

Test size: 1273
{'id': '0', 'question_id': '0', 'document_id': '0', 'question': 'A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?', 'type': 'multiple_choice', 'choices': ['Disclose the error to the patient and put it in the operative report', 'Tell the attending that he cannot fail to disclose this mistake', 'Report the physician to the ethics committee', 'Refuse to dictate the operative report'], 'context': '', 'answer': ['Tell the att

In [4]:
# dataset utility
def get_choices(example):

    if "choices" in example:
        return example["choices"]

    if "options" in example:
        return example["options"]

    raise ValueError("Dataset has no choices/options field")

In [5]:
#Llama-3.1 Load
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Model loaded.")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded.


In [6]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

In [18]:
#gold->A/B/C/D translate(변환)
def get_gold_letter(example):

    labels = ["A", "B", "C", "D"]
    choices = get_choices(example)

    ans = example["answer"]

    # answer가 list인 경우
    if isinstance(ans, list):
        gold_text = ans[0].strip().lower()
    else:
        gold_text = ans.strip().lower()

    for i, choice in enumerate(choices):
        if choice.strip().lower() == gold_text:
            return labels[i]

    return "INVALID"

In [19]:
#Zero-shot_Chat Template Prompt Builder
def build_zero_shot_prompt(example):

    question = example["question"]
    choices = get_choices(example)

    options_text = "\n".join(
        [f"{chr(65+i)}. {choice}" for i, choice in enumerate(choices)]
    )

    prompt = f"""
You are a medical expert solving a USMLE clinical question.

Question:
{question}

Options:
{options_text}

Select the correct answer.

Answer with only one letter: A, B, C, or D.

Answer:
"""

    return prompt

In [20]:
#Zero-shot_Answer Extractor(답변 추출)
#extra_anser: zeroshot,fewshot 둘다 사용
import re

def extract_choice(text):

    text = text.upper()

    match = re.search(r"ANSWER\s*[:\-]?\s*([ABCD])", text)
    if match:
        return match.group(1)

    match = re.search(r"\b([ABCD])\b", text[-100:])
    if match:
        return match.group(1)

    return "INVALID"

In [21]:
#Zero-shot_Inference Function(추론 기능)
import torch
import re
import json
from tqdm import tqdm

def run_inference(data, prompt_builder, save_path=None):
    correct = 0
    results = []

    for example in tqdm(data):
        prompt = prompt_builder(example)

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

        prediction = extract_choice(decoded)
        gold = get_gold_letter(example)

        is_correct = prediction == gold
        if is_correct:
            correct += 1

        results.append({
            "gold": gold,
            "prediction": prediction,
            "correct": is_correct
        })

    accuracy = correct / len(data)

    # 저장 옵션
    if save_path is not None:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=2)

        # metrics 저장
        metrics_path = save_path.replace(".json", "_metrics.txt")
        with open(metrics_path, "w") as f:
            f.write(f"Accuracy: {accuracy}\n")

    return accuracy, results

In [40]:
# Zero-shot_sample 20 test

sample = test_data.select(range(20))

save_path = f"{DEBUG_RESULT_DIR}/MedQA_zeroshot_chat_20.json"

acc_sample, res_sample = run_inference(
    sample,
    build_zero_shot_prompt
)

# 결과 저장
with open(save_path, "w") as f:
    json.dump(res_sample, f, indent=2)

print("Zero-shot Sample Accuracy:", acc_sample)
print("Saved to:", save_path)

100%|███████████████████████████████████████████████████████████████████████████████████| 20/20 [00:04<00:00,  4.73it/s]

Zero-shot Sample Accuracy: 0.5
Saved to: /gpfs/data/oermannlab/gaifl/users/yl14814/medqa_hybrid_project/03_results/debug/llama31/MedQA/MedQA_zeroshot_chat_20.json


In [12]:
print(test_data[0])

{'id': '0', 'question_id': '0', 'document_id': '0', 'question': 'A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?', 'type': 'multiple_choice', 'choices': ['Disclose the error to the patient and put it in the operative report', 'Tell the attending that he cannot fail to disclose this mistake', 'Report the physician to the ethics committee', 'Refuse to dictate the operative report'], 'context': '', 'answer': ['Tell the attending that he c

In [23]:
#Zero-shot_full test
save_path = f"{BASELINE_RESULT_DIR}/MedQA_zeroshot_chat_full.json"

acc_full, res_full = run_inference(
    test_data,
    build_zero_shot_prompt
)

# 결과 저장
with open(save_path, "w") as f:
    json.dump(res_full, f, indent=2)

print("Zero-shot Full Accuracy:", acc_full)
print("Saved to:", save_path)

100%|███████████████████████████████████████████████████████████████████████████████| 1273/1273 [04:36<00:00,  4.60it/s]

Zero-shot Full Accuracy: 0.5860172820109977
Saved to: /gpfs/data/oermannlab/gaifl/users/yl14814/medqa_hybrid_project/03_results/baselines/llama31/MedQA/MedQA_zeroshot_chat_full.json


In [24]:
#CoT Prompt 함수
def build_cot_prompt(example):

    question = example["question"]
    choices = example["choices"]

    options_text = "\n".join(
        [f"{chr(65+i)}. {choice}" for i, choice in enumerate(choices)]
    )

    prompt = f"""
You are a medical expert solving a USMLE clinical question.

Question:
{question}

Options:
{options_text}

Let's think step by step.

After reasoning, choose the correct answer.

Answer:
"""

    return prompt

In [32]:
#CoT 전용 답 추출 함수 (중요)
import re

def extract_cot_answer(text):

    text = text.upper()

    # 1️⃣ "Answer: B"
    match = re.search(r"ANSWER\s*[:\-]?\s*([ABCD])", text)
    if match:
        return match.group(1)

    # 2️⃣ "B. Something"
    match = re.search(r"\b([ABCD])\.\s", text)
    if match:
        return match.group(1)

    # 3️⃣ 마지막 fallback
    match = re.search(r"\b([ABCD])\b", text[-200:])
    if match:
        return match.group(1)

    return "INVALID"

In [33]:
#CoT Batch Inference 함수
from tqdm import tqdm
import torch
import json

def run_cot_inference_batch(data, prompt_builder, save_path=None, batch_size=4):

    correct = 0
    results = []

    prompts = [prompt_builder(data[i]) for i in range(len(data))]

    for i in tqdm(range(0, len(prompts), batch_size)):

        batch_prompts = prompts[i:i+batch_size]

        batch_examples = [
            data[k] for k in range(i, min(i+batch_size, len(data)))
        ]

        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(model.device)

        with torch.inference_mode():

            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id
            )

        for j in range(len(batch_examples)):

            generated_tokens = outputs[j][inputs.input_ids.shape[-1]:]

            decoded = tokenizer.decode(
                generated_tokens,
                skip_special_tokens=True
            ).strip()

            prediction = extract_cot_answer(decoded)

            gold = get_gold_letter(batch_examples[j])

            is_correct = prediction == gold

            if is_correct:
                correct += 1

            results.append({
                "gold": gold,
                "prediction": prediction,
                "correct": is_correct
            })

    accuracy = correct / len(data)

    if save_path:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=2)

    return accuracy, results

In [34]:
#CoT sample test
sample = test_data.select(range(20))

save_path = f"{DEBUG_RESULT_DIR}/MedQA_cot_chat_20.json"

acc_sample, res_sample = run_cot_inference_batch(
    sample,
    build_cot_prompt,
    save_path=save_path,
    batch_size=4
)

print("CoT Accuracy (20):", acc_sample)
print("Saved to:", save_path)

100%|█████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:28<00:00,  5.61s/it]

CoT Accuracy (20): 0.6
Saved to: /gpfs/data/oermannlab/gaifl/users/yl14814/medqa_hybrid_project/03_results/debug/llama31/MedQA/MedQA_cot_chat_20.json


In [28]:
print(res_sample[:5])

[{'gold': 'B', 'prediction': 'D', 'correct': False}, {'gold': 'D', 'prediction': 'C', 'correct': False}, {'gold': 'B', 'prediction': 'A', 'correct': False}, {'gold': 'D', 'prediction': 'A', 'correct': False}, {'gold': 'B', 'prediction': 'C', 'correct': False}]


In [31]:
#디버그용 test 코드
example = test_data[0]

prompt = build_cot_prompt(example)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=120
)

decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(decoded)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



You are a medical expert solving a USMLE clinical question.

Question:
A junior orthopaedic surgery resident is completing a carpal tunnel repair with the department chairman as the attending physician. During the case, the resident inadvertently cuts a flexor tendon. The tendon is repaired without complication. The attending tells the resident that the patient will do fine, and there is no need to report this minor complication that will not harm the patient, as he does not want to make the patient worry unnecessarily. He tells the resident to leave this complication out of the operative report. Which of the following is the correct next action for the resident to take?

Options:
A. Disclose the error to the patient and put it in the operative report
B. Tell the attending that he cannot fail to disclose this mistake
C. Report the physician to the ethics committee
D. Refuse to dictate the operative report

Let's think step by step.

After reasoning, choose the correct answer.

Answer:

In [ ]:
res_sample[:5]

In [35]:
#CoT_full test
save_path = f"{BASELINE_RESULT_DIR}/MedQA_cot_chat_full.json"

acc_full, res_full = run_cot_inference_batch(
    test_data,
    build_cot_prompt,
    save_path=save_path,
    batch_size=8
)

print("CoT Full Accuracy:", acc_full)
print("Saved to:", save_path)

100%|█████████████████████████████████████████████████████████████████████████████████| 160/160 [18:26<00:00,  6.91s/it]

CoT Full Accuracy: 0.6135113904163394
Saved to: /gpfs/data/oermannlab/gaifl/users/yl14814/medqa_hybrid_project/03_results/baselines/llama31/MedQA/MedQA_cot_chat_full.json


In [36]:
#Few-shot Prompt 함수
def build_fewshot_prompt(example):

    few_shot_example = """
Question:
A patient with crushing chest pain radiating to the left arm most likely has:

Options:
A. Pneumonia
B. Myocardial infarction
C. GERD
D. Panic attack

Answer: B
"""

    labels = ["A", "B", "C", "D"]

    options_text = "\n".join(
        [f"{labels[i]}. {choice}" for i, choice in enumerate(example["choices"])]
    )

    messages = [
        {
            "role": "system",
            "content": "You are a highly trained physician solving USMLE medical questions."
        },
        {
            "role": "user",
            "content": f"""
Here is an example USMLE question.

{few_shot_example}

Now answer the following question.

Question:
{example["question"]}

Options:
{options_text}

Answer with only the letter (A, B, C, or D).
"""
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return prompt

In [37]:
#Few-shot_Inference 함수
from tqdm import tqdm
import torch
import json

def run_fewshot_inference(data, prompt_builder, save_path=None):

    correct = 0
    results = []

    for example in tqdm(data):

        # prompt 생성
        prompt = prompt_builder(example)

        # tokenizer → GPU 이동
        inputs = tokenizer(
            prompt,
            return_tensors="pt"
        ).to(model.device)

        # inference
        with torch.inference_mode():

            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                do_sample=False,
                temperature=0.0,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id
            )

        # 생성된 token만 추출
        generated_tokens = outputs[0][inputs.input_ids.shape[-1]:]

        # decoding
        decoded = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        ).strip()

        # prediction 추출
        prediction = extract_cot_answer(decoded)

        # gold answer
        gold = get_gold_letter(example)

        # 정답 여부
        is_correct = prediction == gold

        if is_correct:
            correct += 1

        # 결과 저장
        results.append({
            "gold": gold,
            "prediction": prediction,
            "correct": is_correct
        })

    # accuracy 계산
    accuracy = correct / len(data)

    # 파일 저장
    if save_path:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=2)

    return accuracy, results

In [38]:
#Few-shot_sample 20 test
sample = test_data.select(range(20))

save_path = f"{DEBUG_RESULT_DIR}/MedQA_fewshot_chat_20.json"

acc_sample, res_sample = run_fewshot_inference(
    sample,
    build_fewshot_prompt,
    save_path=save_path
)

print("Few-shot Accuracy (20):", acc_sample)
print("Saved to:", save_path)

100%|███████████████████████████████████████████████████████████████████████████████████| 20/20 [00:01<00:00, 12.47it/s]

Few-shot Accuracy (20): 0.5
Saved to: /gpfs/data/oermannlab/gaifl/users/yl14814/medqa_hybrid_project/03_results/debug/llama31/MedQA/MedQA_fewshot_chat_20.json


In [39]:
#Few-shot_full test 
save_path = f"{BASELINE_RESULT_DIR}/MedQA_fewshot_chat_full.json"

acc_full, res_full = run_fewshot_inference(
    test_data,
    build_fewshot_prompt,
    save_path=save_path
)

print("Few-shot Full Accuracy:", acc_full)
print("Saved to:", save_path)

100%|███████████████████████████████████████████████████████████████████████████████| 1273/1273 [01:56<00:00, 10.94it/s]

Few-shot Full Accuracy: 0.5137470542026709
Saved to: /gpfs/data/oermannlab/gaifl/users/yl14814/medqa_hybrid_project/03_results/baselines/llama31/MedQA/MedQA_fewshot_chat_full.json


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

In [ ]:
# analyze_result.py: 결과 분석
# error_analysis.py: 에러 분석
# cross-model error analysis: 3모델 비교

#1.analyze_result.py 역할: 해당 4개를 자동으로 생성
#→ accuracy table
#→ heatmap
#→ model ranking
#→ dataset ranking

#2.error_analysis.py 역할: hybrid 설계 근거 및 동시에 해당 3개의 파일 생성
#2-1. all_models_wrong 파일 근거:
#→ frontier model 필요
#→ retrieval 필요
#2-2. only_cot_correct 파일 근거:
#→ reasoning task
#2-3. only_fewshot_correct 파일 근거:
#→ pattern learning task

#3.cross-model error analysis 파일 근거: Hybrid routing 모델 설계에 사용, 각각의 모델만 맞춘 문제를 파악 및 3 모델 비교 분석을 자동 생성
#→ 3모델이 어떤 문제를 맞추고 틀렸는지 파악

#총단계: baseline 실행 → 결과 분석 → 에러 분석 → 모델 비교